# Road accident Severity - Model Training

## 1. Data Preparation

In [37]:
# import libs
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
# ML
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

import joblib
import json
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

In [2]:
PROJECT_DIR = Path.cwd().resolve().parents[1]
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

### 1.1. Load Processed Dataset

In [3]:
dataset_file_path = PROCESSED_DATA_DIR.resolve() / "accident_severity_dataset_processed.csv"
df = pd.read_csv(dataset_file_path, low_memory=False)

print(df.shape)
df.head()

(440337, 37)


,catu,grav,sexe,trajet,locp,etatp,catv,obs,obsm,choc,...,surf,infra,situ,vma,year,age,age_unknown,hour,has_safety_equipment,grav_ord
0,1,3,1,5,-1,-1,motorcycle,none,2,1,...,1,0,1,50,2022,14,0,16,1,2.0
1,1,1,1,5,-1,-1,car,none,2,2,...,1,0,1,50,2022,74,0,16,1,0.0
2,1,4,1,9,0,-1,car,none,2,8,...,1,0,1,50,2022,34,0,8,1,1.0
3,1,1,1,4,0,-1,truck,none,2,1,...,1,0,1,50,2022,52,0,8,1,0.0
4,1,1,1,0,-1,-1,car,none,2,1,...,1,5,1,50,2022,20,0,17,1,0.0


### 1.2. Dataset Overview

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 440337 entries, 0 to 440336
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   catu                  440337 non-null  int64  
 1   grav                  440337 non-null  int64  
 2   sexe                  440337 non-null  int64  
 3   trajet                440337 non-null  int64  
 4   locp                  440337 non-null  int64  
 5   etatp                 440337 non-null  int64  
 6   catv                  440337 non-null  str    
 7   obs                   440337 non-null  str    
 8   obsm                  440337 non-null  int64  
 9   choc                  440337 non-null  int64  
 10  manv                  440337 non-null  int64  
 11  motor                 440337 non-null  int64  
 12  jour                  440337 non-null  int64  
 13  mois                  440337 non-null  int64  
 14  lum                   440337 non-null  int64  
 15  agg        

In [5]:
df.describe(include="all").transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
catu,440337.0,NaN,NaN,NaN,1.325369,0.608368,1.0,1.0,1.0,2.0,3.0
grav,440337.0,NaN,NaN,NaN,2.511127,1.382103,-1.0,1.0,3.0,4.0,4.0
sexe,440337.0,NaN,NaN,NaN,1.269587,0.565101,-1.0,1.0,1.0,2.0,2.0
trajet,440337.0,NaN,NaN,NaN,3.133543,2.785088,-1.0,0.0,4.0,5.0,9.0
locp,440337.0,NaN,NaN,NaN,-0.216357,1.25219,-1.0,-1.0,0.0,0.0,9.0
etatp,440337.0,NaN,NaN,NaN,-0.819561,0.631314,-1.0,-1.0,-1.0,-1.0,3.0
catv,440337,10,car,274844,NaN,NaN,NaN,NaN,NaN,NaN,NaN
obs,440337,5,none,373187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
obsm,440337.0,NaN,NaN,NaN,1.611868,1.229436,-1.0,1.0,2.0,2.0,9.0
choc,440337.0,NaN,NaN,NaN,2.847533,2.393336,-1.0,1.0,2.0,4.0,9.0


In [6]:
df.nunique()

catu                      3
grav                      5
sexe                      3
trajet                    8
locp                     11
etatp                     4
catv                     10
obs                       5
obsm                      8
choc                     11
manv                     28
motor                     8
jour                     31
mois                     12
lum                       6
agg                       2
int                      10
atm                      10
col                       8
catr                      8
v1                        4
circ                      5
nbv                      16
vosp                      5
prof                      5
plan                      5
larrout                 137
surf                     10
infra                    11
situ                      8
vma                      39
year                      3
age                     108
age_unknown               2
hour                     24
has_safety_equipment

## 2. Goal


- __Hypothesis__: Can accident serverity be prdicted only using environmental and road conditions?
 
- __Target__: grav_order

- __Predictors__: environmental vars + road characteristics

## 3. Feature Selection

In [ ]:
# Selected features
# grav_ord is already ordinal encoded: 0=indemne, 1=léger, 2=hospitalisé, 3=tué
TARGET = "grav_ord"
FEATURES = [
    "atm",# weather
    "surf",# road surface
    "lum",# lighting
    "infra",# infrastructure
    "situ",# accident location
    "plan",# road profile
    "catr",# road category
    "agg",# urban / rural
    "vma",# speed limit
    "hour",# accident time
    "mois",# seasonality

    # user information
    "age",
    "age_unknown",
    
]
df_model = df.dropna(subset=[TARGET]).copy()
df_model[TARGET] = df_model[TARGET].astype(int)
print(f"Dropped rows (grav_ord NaN): {df.shape[0] - df_model.shape[0]}")
print(df_model[TARGET].value_counts(normalize=True).sort_index())

Dropped rows (grav_ord NaN): 387
grav_ord
0    0.428592
1    0.397025
2    0.148758
3    0.025626
Name: proportion, dtype: float64


In [33]:
# - atm, surf, lum: weather, visibility, adhesion -> main hypoth
# - infra, situ, plan, catr: road improvement
# - agg, vma: speed context
# - hour, mois: saisonality/daily (natural light naturelle, trafic)
# - age, age_unknown: to capture physical vulnerability
assert "grav" not in FEATURES and TARGET not in FEATURES
missing = [c for c in FEATURES if c not in df_model.columns]
print("Missing rows:", missing)  # should be empty

Missing rows: []


docs:
- [pandas docs](https://pandas.pydata.org/docs/user_guide/index.html)
- [scikit-learn - data leakage](https://scikit-learn.org/stable/common_pitfalls.html)

## 2. Train / Test / Validate

In [ ]:
# encoding
nominal_cols = ["atm", "surf", "lum", "infra", "situ", "plan", "catr", "agg"]
numeric_cols = ["vma", "hour", "mois", "age", "age_unknown"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), nominal_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

In [38]:
train_mask = df_model["year"].isin([2022, 2023])
test_mask = df_model["year"] == 2024

X_train = df_model.loc[train_mask, FEATURES]
X_test = df_model.loc[test_mask, FEATURES]
y_train = df_model.loc[train_mask, TARGET]
y_test = df_model.loc[test_mask, TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train distribution:\n", y_train.value_counts(normalize=True).sort_index())
print("Test distribution:\n", y_test.value_counts(normalize=True).sort_index())

Train: (283303, 13), Test: (156647, 13)
Train distribution:
 grav_ord
0    0.428036
1    0.395979
2    0.149945
3    0.026039
Name: proportion, dtype: float64
Test distribution:
 grav_ord
0    0.429596
1    0.398916
2    0.146610
3    0.024878
Name: proportion, dtype: float64


### 2.1. Train

In [40]:
# baseline model
#  logistic regression
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
print("=== Baseline: Logistic Regression ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_baseline):.3f}")
print(f"Macro F1: {f1_score(y_test, y_pred_baseline, average='macro'):.3f}")

=== Baseline: Logistic Regression ===
Balanced Accuracy: 0.402
Macro F1: 0.311


In [41]:
# decision tree
tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(class_weight="balanced", random_state=42)),
])
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)
print("=== Decision Tree (default params) ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_tree):.3f}")

=== Decision Tree (default params) ===
Balanced Accuracy: 0.299


In [42]:
# random forest
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
])
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("=== Random Forest (default params) ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_rf):.3f}")

=== Random Forest (default params) ===
Balanced Accuracy: 0.325


In [43]:
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        objective="multi:softprob", num_class=4, eval_metric="mlogloss", random_state=42
    )),
])
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("=== XGBoost (default params) ===")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_xgb):.3f}")

=== XGBoost (default params) ===
Balanced Accuracy: 0.322


In [ ]:
# grid search
# scoring: balanced_acc -> because classe are highly imbalanced
# very few killed and too much unhurt: cool help to avoid model always predict "unhurt"
models_grids = {
    "logreg": (
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        {"model__C": [0.1, 1, 10]}
    ),
    "decision_tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        {"model__max_depth": [8, 12, 15], "model__min_samples_leaf": [20, 50]}
    ),
    "random_forest": (
        RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        {"model__n_estimators": [100], "model__max_depth": [12, 15]}
    ),
    "xgboost": (
        XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            random_state=42,
            tree_method="hist"
        ),
        {"model__n_estimators": [100, 200], "model__max_depth": [6, 8],
         "model__learning_rate": [0.1]}
    ),
}

search_results = {}
for name, (model, grid) in models_grids.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    search = GridSearchCV(pipe, grid, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=1)
    search.fit(X_train, y_train)
    search_results[name] = search
    print(f"{name}: best CV balanced_accuracy={search.best_score_:.3f}, params={search.best_params_}")

Fitting 5 folds for each of 3 candidates, totalling 15 fits
logreg: best CV balanced_accuracy=0.400, params={'model__C': 10}
Fitting 5 folds for each of 12 candidates, totalling 60 fits
decision_tree: best CV balanced_accuracy=0.397, params={'model__max_depth': 10, 'model__min_samples_leaf': 50}
Fitting 5 folds for each of 6 candidates, totalling 30 fits


KeyboardInterrupt: 

### 2.2. Evaluation / tests

In [ ]:
# metric eval
eval_summary = []
for name, search in search_results.items():
    y_pred = search.predict(X_test)
    eval_summary.append({
        "model": name,
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    })

eval_df = pd.DataFrame(eval_summary).sort_values("balanced_accuracy", ascending=False)
print(eval_df)

best_name = eval_df.iloc[0]["model"]
best_search = search_results[best_name]
best_pipeline = best_search.best_estimator_
print(f"\nModel choosen: {best_name}")

y_pred_best = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_best,
      target_names=["indemne", "léger", "hospitalisé", "tué"]))

In [ ]:
# conf matrix
cm = confusion_matrix(y_test, y_pred_best)
labels = ["indemne", "léger", "hospitalisé", "tué"]

fig_cm = px.imshow(cm, text_auto=True, x=labels, y=labels,
                    color_continuous_scale="Blues",
                    labels=dict(x="Prédit", y="Réel", color="Nb cas"),
                    title=f"Confusion Matrix — {best_name} (test 2024)")
fig_cm.show()

In [ ]:
# features importance
if best_name in ["decision_tree", "random_forest", "xgboost"]:
    ohe_names = best_pipeline.named_steps["preprocessor"] \
                             .named_transformers_["cat"] \
                             .get_feature_names_out(nominal_cols)
    all_feature_names = list(ohe_names) + numeric_cols

    importances = best_pipeline.named_steps["model"].feature_importances_
    fi_df = pd.DataFrame({"feature": all_feature_names, "importance": importances}) \
              .sort_values("importance", ascending=False).head(15)

    fig_fi = px.bar(fi_df, x="importance", y="feature", orientation="h",
                     title=f"Feature importance — {best_name}")
    fig_fi.show()
else:
    # LogisticRegression: coefficients per classe
    ohe_names = best_pipeline.named_steps["preprocessor"] \
                             .named_transformers_["cat"] \
                             .get_feature_names_out(nominal_cols)
    all_feature_names = list(ohe_names) + numeric_cols
    coefs = pd.DataFrame(best_pipeline.named_steps["model"].coef_,
                          columns=all_feature_names, index=labels)
    print(coefs.T.sort_values(labels[-1], ascending=False).head(15))